<a href="https://www.kaggle.com/code/eavprog/absolute-fx-behavioral-profiling-k-means-cluste?scriptVersionId=304298752" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
import numpy as np

# 1. Загрузка
file_path = '/kaggle/input/notebooks/eavprog/abscur2/abscur.csv'
df = pd.read_csv(file_path, parse_dates=['Date'])
df = df.sort_values('Date').set_index('Date')

# 2. Очистка (ML-Ready)
# Удаляем валюты, где вообще нет данных (если такие есть)
df = df.dropna(axis=1, how='all')

# Заполняем пропуски: сначала вперед (последним известным), потом назад (для самого начала истории)
df = df.ffill().bfill()

# Проверка на критические ошибки
if df.isnull().values.any():
    print("⚠️ Внимание: в данных остались пропуски!")
else:
    print("✅ Данные очищены и готовы к анализу.")

print(f"Итоговая матрица: {df.shape[0]} дней x {df.shape[1]} валют")

✅ Данные очищены и готовы к анализу.
Итоговая матрица: 5898 дней x 45 валют


# Этап 1. Подготовка данных и Feature Engineering

На этом этапе мы переходим от анализа временных рядов к формированию матрицы признаков (Features). Наша цель — извлечь из исторических данных абсолютных курсов ключевые поведенческие характеристики каждой валюты, которые станут основой для обучения модели.

### Архитектура данных
Для каждой из 45+ валют мы рассчитываем вектор признаков, описывающий её «характер» на протяжении последних 20 лет. Мы используем четыре метрики, охватывающие риск, доходность и устойчивость:

1.  **Volatility (Волатильность)** — годовое стандартное отклонение логарифмических доходностей. Показывает уровень рыночного «шума» и неопределенности.
2.  **CAGR (Среднегодовой темп роста)** — геометрическая средняя доходность. Позволяет отличить укрепляющиеся валюты от девальвационных.
3.  **Maximum Drawdown (Максимальная просадка)** — детектор системной хрупкости, показывающий худший сценарий обесценения актива.
4.  **Recovery Speed (Скорость восстановления)** — временной фактор, указывающий, насколько быстро рынок восстанавливает доверие к активу после кризиса.

### Используемый инструментарий
* **Библиотеки**: `Pandas` и `NumPy` для векторных вычислений, `Scikit-learn` для последующего обучения, `IPython.display` для интерактивных таблиц.
* **Интерактивность**: Каждый тикер в итоговой таблице будет представлен ссылкой на интерактивный график абсолютного курса на сайте проекта [Abscur](https://www.abscur.ru), что позволяет визуально верифицировать результаты кластеризации.

Результатом этого этапа станет матрица признаков `features_df`, полностью готовая к этапу масштабирования и применения алгоритма **K-Means**.

## 1.1. Анализ изменчивости логарифмической доходности (Volatility)

На первом этапе подготовки данных мы рассчитываем **годовую волатильность логарифмических доходностей**. 

### Почему это важно для машинного обучения?
В отличие от анализа номинальных курсов, работа с доходностями позволяет алгоритму **K-Means** сравнивать активы разного порядка стоимости. Логарифмирование нормализует распределение, сглаживая экстремальные выбросы, что критично для корректной работы метрических алгоритмов кластеризации.

**Технические детали расчета:**
* Мы используем стандартное отклонение ежедневных логарифмических доходностей.
* Результат масштабируется к годовому значению (умножение на $\sqrt{252}$), что позволяет интерпретировать риск в привычных для финансового анализа величинах.
* **Важное отличие:** На сайте проекта представлена страница [Рейтинги волатильности](https://www.abscur.ru/p/blog-page_26.html), где используется расчет на основе *арифметической* доходности без приведения к годовому значению. Здесь же мы намеренно переходим к годовой логарифмической волатильности, чтобы подготовить данные для обучения модели.

В таблицах ниже представлены крайние значения выборки. Тикеры являются кликабельными и ведут на интерактивные графики соответствующих абсолютных курсов на сайте проекта **Abscur**.

In [2]:
import numpy as np
import pandas as pd
from IPython.display import HTML

# Создаем базовый DataFrame для признаков (индекс — тикеры валют)
features_df = pd.DataFrame(index=df.columns)

# 1. Вычисляем ежедневные логарифмические доходности
log_returns = np.log(df / df.shift(1))

# 2. Считаем стандартное отклонение и годовое масштабирование
features_df['Volatility'] = log_returns.std() * np.sqrt(252)

# Универсальная функция для создания ссылок
def make_clickable(ticker):
    url = f"https://www.abscur.ru/p/2.html?abs={ticker}"
    return f'<a href="{url}" target="_blank">{ticker}</a>'

# Универсальная функция для вывода результатов с ПРАВИЛЬНОЙ сортировкой
def display_sorted_results(df_input, column, ascending=True, title=""):
    # Сначала сортируем ЧИСЛА, берем 5 строк, и делаем копию
    sorted_df = df_input[[column]].sort_values(by=column, ascending=ascending).head(5).copy()
    
    # Только теперь преобразуем число в красивую строку с процентами
    sorted_df[column] = (sorted_df[column] * 100).round(2).astype(str) + '%'
    
    # Делаем ссылки в индексе
    sorted_df.index = [make_clickable(t) for t in sorted_df.index]
    
    print(title)
    display(HTML(sorted_df.to_html(escape=False)))

# Вывод результатов
display_sorted_results(features_df, 'Volatility', ascending=True, 
                       title="Топ-5 валют с МИНИМАЛЬНОЙ изменчивостью доходности:")

display_sorted_results(features_df, 'Volatility', ascending=False, 
                       title="Топ-5 валют с МАКСИМАЛЬНОЙ изменчивостью доходности:")

Топ-5 валют с МИНИМАЛЬНОЙ изменчивостью доходности:


,Volatility
HKD,2.23%
SGD,2.51%
ILS,2.86%
KWD,3.11%
USD,3.17%


Топ-5 валют с МАКСИМАЛЬНОЙ изменчивостью доходности:


,Volatility
ARS,19.95%
UAH,16.08%
EGP,15.45%
RUB,12.09%
TRY,11.38%


## 1.2. Расчет среднегодового темпа роста (CAGR)

Второй ключевой признак для нашей модели — **CAGR (Compound Annual Growth Rate)**. Это геометрическая средняя доходность, которая показывает темп роста стоимости валюты, как если бы она росла с постоянной скоростью в течение всего периода исследования.

### Зачем CAGR нужен модели?
В отличие от простой средней доходности, CAGR учитывает эффект накопления (сложный процент). Это позволяет алгоритму **K-Means** более точно разделять активы на:
1. **«Стабильно растущие»** (валюты с положительным CAGR, увеличивающие свою покупательную способность).
2. **«Девальвационные»** (валюты, демонстрирующие системное снижение ценности относительно мировой корзины).

**Методология расчета:**
* Формула: $(P_{end} / P_{start})^{1/years} - 1$.
* Данные рассчитываются на основе всего доступного исторического диапазона (с 2006 года).
* Итоговое значение приводится к годовому формату в процентах, что делает его сопоставимым с волатильностью.

In [3]:
# --- РАСЧЕТ CAGR ---

# 1. Считаем временной интервал в годах
total_days = (df.index.max() - df.index.min()).days
years = total_days / 365.25

# 2. Вычисляем CAGR и сохраняем в основной DataFrame признаков
features_df['CAGR'] = (df.iloc[-1] / df.iloc[0])**(1/years) - 1

# 3. Функция для форматирования и вывода (сортируем ЧИСЛА, а не текст)
def display_sorted_results(df_input, column, ascending=False, title=""):
    # Сортируем исходные числа
    sorted_df = df_input[[column]].sort_values(by=column, ascending=ascending).head(5).copy()
    
    # Только после сортировки преобразуем в текст для красоты
    sorted_df[column] = (sorted_df[column] * 100).round(2).astype(str) + '%'
    
    # Добавляем ссылки на сайт
    sorted_df.index = [make_clickable(t) for t in sorted_df.index]
    
    print(title)
    display(HTML(sorted_df.to_html(escape=False)))

print(f"Анализируемый период: {years:.1f} лет\n")

# Вывод лидеров роста
display_sorted_results(features_df, 'CAGR', ascending=False, 
                       title="Топ-5 валют с максимальным темпом роста (CAGR):")

# Вывод лидеров падения (девальвации)
display_sorted_results(features_df, 'CAGR', ascending=True, 
                       title="Топ-5 валют с максимальным темпом снижения (девальвация):")

Анализируемый период: 19.6 лет

Топ-5 валют с максимальным темпом роста (CAGR):


,CAGR
CHF,3.06%
CNY,2.91%
ILS,2.56%
CZK,2.44%
THB,2.31%


Топ-5 валют с максимальным темпом снижения (девальвация):


,CAGR
ARS,-21.16%
TRY,-12.12%
EGP,-7.62%
UAH,-3.39%
PKR,-2.86%


## 1.3. Расчет максимальной просадки (Maximum Drawdown)

Третий критический параметр для обучения модели — **Maximum Drawdown (MDD)**. Эта метрика фиксирует «худший сценарий» (максимальный исторический обвал) для каждой валюты за весь период исследования.

### Роль MDD в кластеризации
Для алгоритма **K-Means** значение просадки служит индикатором системной хрупкости. В сочетании с волатильностью, MDD позволяет эффективно отделять:
* **«Защитные активы»**: Валюты, чьи просадки носят поверхностный и краткосрочный характер.
* **«Зоны системного риска»**: Валюты, пережившие обвалы на 50% и более, что часто свидетельствует о структурных проблемах в экономике или смене валютных режимов.

**Методология:**
* Расчет ведется от текущего исторического максимума (*High Water Mark*) до локального минимума.
* Значение выражается в процентах. Чем ближе показатель к -100%, тем сильнее было обесценение актива относительно мировой корзины валют в его худшей точке.

In [4]:
# --- РАСЧЕТ MAXIMUM DRAWDOWN (MDD) ---

# 1. Рассчитываем исторический максимум (High Water Mark) на каждый момент времени
rolling_max = df.cummax()

# 2. Вычисляем текущую просадку в процентах от пика
drawdowns = (df - rolling_max) / rolling_max

# 3. Находим минимальное значение просадки (самую глубокую точку) для каждой валюты
features_df['Max_Drawdown'] = drawdowns.min()

# 4. Вывод результатов с использованием нашей функции (сортируем от самых глубоких падений)
display_sorted_results(features_df, 'Max_Drawdown', ascending=True, 
                       title="Топ-5 валют с САМЫМИ ГЛУБОКИМИ просадками (MDD):")

display_sorted_results(features_df, 'Max_Drawdown', ascending=False, 
                       title="Топ-5 валют с МИНИМАЛЬНЫМИ просадками:")

Топ-5 валют с САМЫМИ ГЛУБОКИМИ просадками (MDD):


,Max_Drawdown
ARS,-99.15%
TRY,-92.31%
EGP,-80.59%
PKR,-57.06%
KZT,-55.47%


Топ-5 валют с МИНИМАЛЬНЫМИ просадками:


,Max_Drawdown
KWD,-6.72%
CNY,-7.39%
ILS,-7.57%
SGD,-8.18%
HKD,-8.22%


## 1.4. Расчет скорости восстановления (Recovery Speed)

Четвертый признак — **Recovery Speed** — добавляет в наше исследование фактор времени. Мы измеряем среднее количество календарных дней, которое требуется валюте, чтобы вернуться к своему историческому максимуму после начала снижения.

### Почему это важно для машинного обучения?
Этот показатель позволяет алгоритму **K-Means** оценить «вязкость» кризисов для каждого актива. 
* **Защитные активы**: характеризуются коротким циклом восстановления. Даже если происходит волатильное движение вниз, рынок быстро выкупает просадку, возвращая актив к росту.
* **Зоны девальвации**: могут иметь умеренную волатильность, но катастрофически долгое время восстановления. Если валюта годами не может обновить максимум, она становится токсичной для долгосрочного инвестора.

**Методология:**
* Программа идентифицирует все периоды, когда текущий курс был ниже зафиксированного ранее исторического пика (*High Water Mark*).
* Рассчитывается средняя длительность таких периодов для каждой валюты.
* Чем меньше значение, тем выше «эластичность» валюты и доверие к ней со стороны глобального рынка.

In [5]:
# --- РАСЧЕТ RECOVERY SPEED ---

def calculate_recovery_days(series):
    # Находим периоды, когда цена ниже исторического максимума (Underwater)
    is_underwater = series < series.cummax()
    
    # Группируем последовательные дни "под водой"
    # (каждый раз, когда состояние меняется, создается новая группа)
    underwater_groups = (is_underwater != is_underwater.shift()).cumsum()
    
    # Считаем длительность каждой такой группы
    # Нас интересуют только те группы, где is_underwater == True
    recovery_periods = is_underwater[is_underwater].groupby(underwater_groups[is_underwater]).count()
    
    # Возвращаем среднее количество дней нахождения в просадке
    # Если просадок не было (теоретически), возвращаем 0
    return recovery_periods.mean() if not recovery_periods.empty else 0

# Применяем функцию к каждой колонке (валюте)
features_df['Recovery_Days'] = df.apply(calculate_recovery_days)

# Вывод результатов
# Здесь мы не используем проценты, так как это дни
def display_days_results(df_input, column, ascending=True, title=""):
    sorted_df = df_input[[column]].sort_values(by=column, ascending=ascending).head(5).copy()
    sorted_df[column] = sorted_df[column].round(1).astype(str) + ' дн.'
    sorted_df.index = [make_clickable(t) for t in sorted_df.index]
    print(title)
    display(HTML(sorted_df.to_html(escape=False)))

display_days_results(features_df, 'Recovery_Days', ascending=True, 
                      title="Топ-5 валют с САМЫМ БЫСТРЫМ восстановлением (в днях):")

display_days_results(features_df, 'Recovery_Days', ascending=False, 
                      title="Топ-5 валют с САМЫМ ДОЛГИМ восстановлением (в днях):")

Топ-5 валют с САМЫМ БЫСТРЫМ восстановлением (в днях):


,Recovery_Days
HKD,25.2 дн.
CNY,26.2 дн.
SGD,27.1 дн.
ILS,29.7 дн.
CHF,33.2 дн.


Топ-5 валют с САМЫМ ДОЛГИМ восстановлением (в днях):


,Recovery_Days
COP,735.4 дн.
UAH,735.4 дн.
TRY,653.4 дн.
BRL,587.8 дн.
ARS,451.5 дн.


## 1.5. Формирование финальной матрицы признаков

На этом шаге мы завершаем этап **Feature Engineering**, объединяя все рассчитанные поведенческие характеристики в единую структуру данных — матрицу признаков. Это финальный аккорд подготовки, превращающий сырые временные ряды в компактный набор векторов, готовых для обучения модели.

**Итоговый состав матрицы:**
* **Volatility**: Характеризует риск и «шумность» доходности актива.
* **CAGR**: Отражает долгосрочную эффективность и темп роста ценности.
* **Max_Drawdown**: Служит детектором системной хрупкости и устойчивости к кризисам.
* **Recovery_Days**: Показывает эластичность валюты и скорость возврата доверия инвесторов.

**Техническая обработка:**
1. **Консолидация**: Данные собраны в единый DataFrame `features_df` с индексацией по тикерам.
2. **Очистка**: Проведена проверка на наличие бесконечных значений (`inf`) и пропусков (`NaN`), которые могли возникнуть из-за специфики котировок некоторых валют.
3. **Валидация**: Сформирована выборка из 45 активов, каждый из которых теперь описан четырехмерным вектором признаков.

Эта таблица станет входными данными для следующего этапа — препроцессинга (масштабирования) и непосредственного поиска кластеров методом **K-Means**.

In [6]:
import numpy as np
import pandas as pd
from IPython.display import HTML

# --- 1. ПОЛНЫЙ РАСЧЕТ ВСЕХ ПРИЗНАКОВ ---

# Создаем чистый DataFrame для признаков
features_df = pd.DataFrame(index=df.columns)

# А. Изменчивость доходности (Volatility)
log_returns = np.log(df / df.shift(1))
features_df['Volatility'] = log_returns.std() * np.sqrt(252)

# Б. Среднегодовой темп роста (CAGR)
total_days = (df.index.max() - df.index.min()).days
years = total_days / 365.25
features_df['CAGR'] = (df.iloc[-1] / df.iloc[0])**(1/years) - 1

# В. Максимальная просадка (Max_Drawdown)
rolling_max = df.cummax()
drawdowns = (df - rolling_max) / rolling_max
features_df['Max_Drawdown'] = drawdowns.min()

# Г. Скорость восстановления (Recovery_Days)
# (Используем функцию calculate_recovery_days, определенную на шаге 1.4)
features_df['Recovery_Days'] = df.apply(calculate_recovery_days)

# --- 2. САНИТАРНАЯ ОБРАБОТКА ---

# Заменяем бесконечности (могут возникнуть при делении на 0) на NaN
features_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Удаляем валюты, по которым не удалось рассчитать хотя бы один признак
features_df.dropna(inplace=True)

# --- 3. ФОРМАТИРОВАННЫЙ ВЫВОД ---

def display_final_matrix(df_input):
    display_df = df_input.copy()
    
    # Сортируем по CAGR для наглядности (самые сильные сверху)
    display_df = display_df.sort_values('CAGR', ascending=False)
    
    # Форматируем данные для отображения
    display_df['Volatility'] = (display_df['Volatility'] * 100).round(2).astype(str) + '%'
    display_df['CAGR'] = (display_df['CAGR'] * 100).round(2).astype(str) + '%'
    display_df['Max_Drawdown'] = (display_df['Max_Drawdown'] * 100).round(2).astype(str) + '%'
    display_df['Recovery_Days'] = display_df['Recovery_Days'].round(1).astype(str) + ' дн.'
    
    # Делаем тикеры ссылками
    display_df.index = [make_clickable(t) for t in display_df.index]
    
    print(f"Итоговая матрица готова к ML: {display_df.shape[0]} валют.")
    display(HTML(display_df.head(15).to_html(escape=False)))

# Запуск отображения
display_final_matrix(features_df)

Итоговая матрица готова к ML: 45 валют.


,Volatility,CAGR,Max_Drawdown,Recovery_Days
CHF,3.96%,3.06%,-12.54%,33.2 дн.
CNY,3.68%,2.91%,-7.39%,26.2 дн.
ILS,2.86%,2.56%,-7.57%,29.7 дн.
CZK,4.68%,2.44%,-11.55%,42.7 дн.
THB,3.81%,2.31%,-9.64%,42.6 дн.
TWD,4.48%,2.29%,-10.65%,41.5 дн.
USD,3.17%,2.24%,-8.49%,47.2 дн.
AED,3.17%,2.24%,-8.49%,47.2 дн.
SAR,3.18%,2.24%,-8.43%,46.8 дн.
QAR,4.24%,2.22%,-9.65%,46.9 дн.
